**Objetivo**

Promover as metas de alfabetização por município da camada Bronze para a Silver.

**Fonte de dados**

- `bronze.meta_alfabetizacao_municipio`

**Destino**

- `silver.meta_alfabetizacao_municipio`

**Granularidade**

- Uma linha por `ano`, `id_municipio` e `rede`.

> As validações de qualidade são informativas, como no notebook de município. Apenas a ausência de colunas obrigatórias impede tecnicamente a execução.

## 0. Configurando sessão Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_meta_alfabetizacao_municipio")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_bronze_meta = f"{par_source_project}.bronze.meta_alfabetizacao_municipio"
par_source_silver_meta = f"{par_source_project}.silver.meta_alfabetizacao_municipio"

colunas_meta = [f"meta_alfabetizacao_{ano}" for ano in range(2024, 2031)]
colunas_percentuais = ["taxa_alfabetizacao", *colunas_meta, "percentual_participacao"]
colunas_esperadas = [
    "ano", "id_municipio", "rede", "taxa_alfabetizacao",
    *colunas_meta, "nivel_alfabetizacao", "percentual_participacao",
    "_ingestao_timestamp", "_fonte"
]

redes_conhecidas = ["Federal", "Estadual", "Municipal", "Privada", "Pública"]

## 3. Leitura dos dados da origem

In [5]:
df_src_meta = (
    spark.read.format("bigquery")
    .option("table", par_source_bronze_meta)
    .load()
)

### 3.1. Validação do contrato de entrada

In [6]:
colunas_ausentes = sorted(set(colunas_esperadas) - set(df_src_meta.columns))
if colunas_ausentes:
    raise ValueError(f"Schema inválido. Colunas ausentes na Bronze: {colunas_ausentes}")

df_meta = df_src_meta.select(*colunas_esperadas)

### 3.2. Diagnóstico dos domínios

In [7]:
for coluna in ["ano", "rede", "nivel_alfabetizacao"]:
    print(f"=== Domínio de {coluna} ===")
    (
        df_meta.groupBy(coluna).count()
        .orderBy(F.desc("count"), F.asc_nulls_first(coluna))
        .show(100, truncate=False)
    )

=== Domínio de ano ===


+----+-----+
|ano |count|
+----+-----+
|2023|5352 |
|2024|5352 |
+----+-----+

=== Domínio de rede ===
+---------+-----+
|rede     |count|
+---------+-----+
|Municipal|10704|
+---------+-----+

=== Domínio de nivel_alfabetizacao ===
+-------------------+-----+
|nivel_alfabetizacao|count|
+-------------------+-----+
|5                  |2059 |
|3                  |1898 |
|4                  |1810 |
|2                  |1783 |
|0                  |1605 |
|1                  |1429 |
|NULL               |120  |
+-------------------+-----+



## 4. Transformações

In [8]:
id_municipio_limpo = F.trim(F.col("id_municipio").cast("string"))
rede_limpa = F.trim(F.col("rede").cast("string"))
rede_minuscula = F.lower(rede_limpa)

df_silver_meta = (
    df_meta
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn(
        "id_municipio",
        F.when(
            id_municipio_limpo.rlike("^[0-9]{1,7}$"),
            F.lpad(id_municipio_limpo, 7, "0")
        ).otherwise(id_municipio_limpo)
    )
    .withColumn(
        "rede",
        F.when(rede_minuscula.isin("pública", "publica", "p�blica", "pãºblica"), F.lit("Pública"))
        .when(rede_limpa == "", F.lit(None))
        .otherwise(F.initcap(rede_minuscula))
    )
    .withColumn("nivel_alfabetizacao", F.col("nivel_alfabetizacao").cast("int"))
    .withColumn("_ingestao_timestamp", F.col("_ingestao_timestamp").cast("timestamp"))
    .withColumn("_fonte", F.trim(F.col("_fonte")))
)

for coluna in colunas_percentuais:
    df_silver_meta = df_silver_meta.withColumn(coluna, F.col(coluna).cast("double"))

### 4.1. Data de carregamento e exclusão de duplicadas

In [9]:
chave = ["ano", "id_municipio", "rede"]
df_silver_meta_antes_dedup = df_silver_meta
df_silver_meta = (
    df_silver_meta
    .dropDuplicates(chave)
    .withColumn("_silver_timestamp", F.current_timestamp())
)

## 5. Validação da qualidade

In [12]:
print("=== Relatório de Qualidade — silver.meta_alfabetizacao_municipio ===")

qtd_bronze = df_src_meta.count()
qtd_silver = df_silver_meta.count()

# 1) Duplicidade na chave natural
dups_antes = (
    df_silver_meta_antes_dedup.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
dups_depois = (
    df_silver_meta.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
print(f"Chaves duplicadas antes da deduplicação: {dups_antes}")
print(f"Chaves duplicadas após deduplicação: {dups_depois}")

# 2) Formato do município e domínios categóricos
id_municipio_invalido = df_silver_meta.filter(
    F.col("id_municipio").isNotNull()
    & ~F.col("id_municipio").rlike("^[0-9]{7}$")
).count()
nivel_invalido = df_silver_meta.filter(
    F.col("nivel_alfabetizacao").isNotNull()
    & ~F.col("nivel_alfabetizacao").isin(0, 1, 2, 3, 4, 5)
).count()
print(f"id_municipio fora do padrão de 7 dígitos: {id_municipio_invalido}")
print(f"nivel_alfabetizacao fora de [0, 5]: {nivel_invalido}")

print("Redes não reconhecidas:")
(
    df_silver_meta.filter(F.col("rede").isNotNull() & ~F.col("rede").isin(*redes_conhecidas))
    .groupBy("rede").count().orderBy(F.desc("count"))
    .show(100, truncate=False)
)

# 3) Nulos nas colunas
for coluna in colunas_esperadas:
    quantidade = df_silver_meta.filter(F.col(coluna).isNull()).count()
    print(f"Nulos em '{coluna}': {quantidade}")

# 4) Percentuais fora da faixa esperada [0, 100]
for coluna in colunas_percentuais:
    quantidade = df_silver_meta.filter(
        F.col(coluna).isNotNull()
        & ((F.col(coluna) < 0) | (F.col(coluna) > 100) | F.isnan(coluna))
    ).count()
    print(f"Valores preenchidos fora de [0, 100] ou NaN em '{coluna}': {quantidade}")

# 5) As metas anuais devem permanecer iguais ou crescer ao longo do horizonte
# Regra de negócio especificada pelo grupo,
# pode ser flexibilizado a depender de definições de negócio diferentes
meta_nao_monotona = df_silver_meta.filter(
    (F.col("meta_alfabetizacao_2025") < F.col("meta_alfabetizacao_2024"))
    | (F.col("meta_alfabetizacao_2026") < F.col("meta_alfabetizacao_2025"))
    | (F.col("meta_alfabetizacao_2027") < F.col("meta_alfabetizacao_2026"))
    | (F.col("meta_alfabetizacao_2028") < F.col("meta_alfabetizacao_2027"))
    | (F.col("meta_alfabetizacao_2029") < F.col("meta_alfabetizacao_2028"))
    | (F.col("meta_alfabetizacao_2030") < F.col("meta_alfabetizacao_2029"))
).count()
print(f"Linhas com metas decrescentes: {meta_nao_monotona}")

# 6) Taxa, nível e participação devem estar preenchidos ou nulos em conjunto
qtd_campos_resultado_nulos = (
    F.when(F.col("taxa_alfabetizacao").isNull(), 1).otherwise(0)
    + F.when(F.col("nivel_alfabetizacao").isNull(), 1).otherwise(0)
    + F.when(F.col("percentual_participacao").isNull(), 1).otherwise(0)
)
resultados_parcialmente_nulos = df_silver_meta.filter(
    (qtd_campos_resultado_nulos > 0) & (qtd_campos_resultado_nulos < 3)
).count()
print(
    "Linhas com nulidade inconsistente entre taxa, nível e participação: "
    f"{resultados_parcialmente_nulos}"
)

# 7) Completude das metas por ano de referência
print("Nulos das metas por ano de referência:")
(
    df_silver_meta.groupBy("ano").agg(
        *[
            F.sum(F.when(F.col(coluna).isNull(), 1).otherwise(0)).alias(f"nulos_{coluna}")
            for coluna in colunas_meta
        ]
    ).orderBy("ano").show(truncate=False)
)

print(f"Linhas Bronze: {qtd_bronze} -> Linhas Silver: {qtd_silver}")

=== Relatório de Qualidade — silver.meta_alfabetizacao_municipio ===
Chaves duplicadas antes da deduplicação: 0
Chaves duplicadas após deduplicação: 0
id_municipio fora do padrão de 7 dígitos: 0
nivel_alfabetizacao fora de [0, 5]: 0
Redes não reconhecidas:
+----+-----+
|rede|count|
+----+-----+
+----+-----+

Nulos em 'ano': 0
Nulos em 'id_municipio': 0
Nulos em 'rede': 0
Nulos em 'taxa_alfabetizacao': 120
Nulos em 'meta_alfabetizacao_2024': 240
Nulos em 'meta_alfabetizacao_2025': 0
Nulos em 'meta_alfabetizacao_2026': 0
Nulos em 'meta_alfabetizacao_2027': 0
Nulos em 'meta_alfabetizacao_2028': 0
Nulos em 'meta_alfabetizacao_2029': 0
Nulos em 'meta_alfabetizacao_2030': 0
Nulos em 'nivel_alfabetizacao': 120
Nulos em 'percentual_participacao': 120
Nulos em '_ingestao_timestamp': 0
Nulos em '_fonte': 0
Valores preenchidos fora de [0, 100] ou NaN em 'taxa_alfabetizacao': 0
Valores preenchidos fora de [0, 100] ou NaN em 'meta_alfabetizacao_2024': 0
Valores preenchidos fora de [0, 100] ou NaN e

## 6. Armazenamento no BigQuery

In [11]:
(
    df_silver_meta.write.format("bigquery")
    .option("table", par_source_silver_meta)
    .option("writeMethod", "direct")
    .option("clusteredFields", "ano,id_municipio,rede")
    .mode("overwrite")
    .save()
)

26/08/24 02:08:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                